# Data Challenge · Transforma datos en decisiones

**Clase demo · Maestría en Ciencia de Datos · Universidad Pontificia Bolivariana**  
**Databricks Free Edition · DANE–SIPSA · Enero a octubre de 2026**

Un negocio de alimentos necesita elegir en qué mercado abastecerse. Hoy vas a **explorar precios, construir gráficas y recomendar una decisión con evidencia**. No necesitas saber programar. El código del caso guiado ya está preparado.

### Antes de empezar

1. Crea tu cuenta en [Databricks Free Edition](https://www.databricks.com/learn/free-edition) o inicia sesión. Elige **Free Edition**, no **Free Trial**.
2. En **Workspace**, abre tu carpeta personal en **Users**. Desde el menú **⋮ → Import**, importa el notebook y el archivo **`SIPSA_Precios_Mayoristas_2026.csv`** en la misma carpeta. Conserva el nombre del CSV.
3. Abre el notebook. Si debes seleccionar el tipo de cómputo, elige **Serverless**. Ejecuta las celdas con **▶** o **Shift + Enter** y sigue las pausas del docente. No uses **Run all** al inicio.

## 1 · Conoce Databricks

**Workspace** es el espacio donde guardas los archivos. El **notebook** es el documento en el que se combinan instrucciones, código y resultados. **Serverless** es el cómputo que Databricks administra para ejecutar las celdas.

- **Markdown:** es el texto formateado que estás leyendo. Haz doble clic sobre una celda de texto para ver cómo se edita. hola
- **Python:** ejecuta la siguiente celda para calcular cuánto cuestan 3 kilos a 2.000 COP por kilo.
- **SQL:** la celda que sigue hace el mismo cálculo con un `SELECT`.

Los dos ejemplos son independientes de los datos de SIPSA.

In [0]:
print("Si un kilo cuesta 2.000 COPS, ¿Cuánto cuestan 3 kilos?")
print("Tres kilos cuestan: ", 3 * 2000, "COPS")

In [0]:
%sql
SELECT 'Costo de 3 kilos' AS mensaje, 3 * 2000 AS costo_en_pesos

## 2 · Carga y explora SIPSA

Usaremos **46.268 observaciones de enero a octubre de 2026** de la serie mensual de precios mayoristas del DANE. Abre el **CSV preparado para el taller**, no el Excel original.

Ejecuta la siguiente celda para abrir el CSV con **pandas** y ver sus primeras 10 filas.

In [0]:
import pandas as pd

sipsa = pd.read_csv("SIPSA_Precios_Mayoristas_2026.csv")
display(sipsa)

### ¿Qué representa cada fila?

Cada fila indica el **precio mayorista promedio de un producto en un mercado durante un mes**.

| Columna | Cómo leerla |
|---|---|
| `mes` | Periodo reportado. `2026-01-01` identifica enero, no una medición del 1 de enero. |
| `producto` y `grupo` | Alimento y familia a la que pertenece. |
| `mercado`, `municipio`, `departamento` | Lugar al que corresponde el precio. Un municipio puede tener más de un mercado. |
| `precio_promedio_cop` y `unidad_precio` | Precio promedio publicado y su unidad. En los ejemplos de clase y el reto, los precios están en **COP/kg**. |

### Crea tu primera gráfica

Observa cómo cambió el precio de **la zanahoria en Corabastos, Bogotá**. Primero seleccionamos ese producto y ese mercado. Luego mostramos el mes y el precio.

In [0]:
serie_practica = sipsa.query(
    'producto == "Zanahoria" and '
    'mercado == "Bogotá, D.C., Corabastos"'
)

display(serie_practica[["mes", "precio_promedio_cop"]].sort_values("mes"))

En el resultado de la celda, pulsa **+** junto a la tabla y elige **Visualization**.

| Ajuste | Selecciona |
|---|---|
| **Visualization Type** | **Line** |
| **X column** | `mes` |
| **Y column** | `precio_promedio_cop` |


**Lee la gráfica.** ¿En qué mes fue más alto el precio? ¿Subió todos los meses o hubo caídas?

### Opcional · Prueba con barras

Crea otra visualización del mismo resultado y selecciona **Bar** con las mismas columnas. **¿Cuál de las dos gráficas te ayuda más a ver el cambio de un mes a otro?**

## 3 · De la gráfica a una decisión de compra

Compararemos **papa capira en Antioquia**. En este CSV, los mismos **12 mercados** tienen un precio en COP/kg para cada uno de los diez meses.

### Pregunta 1 · ¿Dónde habrías comprado en enero de 2026?

Ejecuta las dos celdas siguientes. La primera filtra por producto y departamento. La segunda se queda con enero.

**Crea la gráfica** a partir del resultado de la segunda celda. Elige **Bar**, usa `mercado` como categoría (X) y `precio_promedio_cop` como valor (Y). Si está disponible, activa **Horizontal chart**. Si necesitas elegir una agregación, usa **Average**, pues hay una fila por mercado. Guarda la gráfica y señala **qué mercado elegirías y por cuánto**.

In [0]:
# Filtramos la papa capira en Antioquia.
papa_capira_antioquia = sipsa.query(
    'producto == "Papa capira" and departamento == "ANTIOQUIA"'
)
display(papa_capira_antioquia[["mes", "mercado", "precio_promedio_cop"]])

In [0]:
# Nos quedamos con los precios de enero.
enero = papa_capira_antioquia.query('mes == "2026-01-01"')
display(enero[["mercado", "precio_promedio_cop"]].sort_values("precio_promedio_cop"))

### Pregunta 2 · ¿Y si tuvieras que comprar en el mismo mercado de enero a octubre?

Imagina que compras **la misma cantidad cada mes** y no cambias de mercado. Antes de ejecutar la celda, piensa si mantendrías tu elección de enero.

Calcula el promedio de los diez precios mensuales de cada mercado y crea una segunda gráfica de barras con las mismas columnas.
**¿Qué hace `as_index=False`?** Mantiene `mercado` como columna para que puedas seleccionarla al crear la gráfica. No cambia el promedio.

In [0]:
# Calculamos un precio promedio de enero a octubre por mercado.
promedio_periodo = papa_capira_antioquia.groupby(
    "mercado", as_index=False
)["precio_promedio_cop"].mean()

display(promedio_periodo.sort_values("precio_promedio_cop").round(2))

### Compara los gastos estimados

El precio de enero sirve para **elegir** el primer mercado. Para calcular el gasto de **ambas** opciones usamos el precio promedio de enero a octubre.

Si compras **500 kg al mes durante 10 meses**, compras **5.000 kg** en total. Multiplica el precio promedio de cada mercado por esos kilos y resta los dos gastos. 

In [0]:
# Identificamos el mercado más barato de enero y el de todo el periodo.
mercado_enero = enero.sort_values("precio_promedio_cop").iloc[0]["mercado"]
mejor_periodo = promedio_periodo.sort_values("precio_promedio_cop").iloc[0]

# Buscamos el promedio de enero a octubre del mercado que elegimos en enero.
precio_periodo_del_ganador_de_enero = promedio_periodo.loc[
    promedio_periodo["mercado"] == mercado_enero, "precio_promedio_cop"
].iloc[0]

kilos_totales = 500 * 10
gasto_enero = round(precio_periodo_del_ganador_de_enero * kilos_totales)
gasto_periodo = round(mejor_periodo["precio_promedio_cop"] * kilos_totales)

print(f"Si mantenemos {mercado_enero}: {gasto_enero:,} COP".replace(",", "."))
print(f"Si elegimos {mejor_periodo['mercado']}: {gasto_periodo:,} COP".replace(",", "."))
print(f"Diferencia estimada: {gasto_enero - gasto_periodo:,} COP".replace(",", "."))

## 4 · ¿Y si los datos no caben en tu equipo?

**Demostración opcional.** El CSV del taller puede analizarse en una sola máquina con pandas. Si un histórico no cabe en el **disco del portátil**, puede guardarse en la nube y consultarse desde Databricks sin descargarlo completo. Si el **procesamiento** supera la capacidad de una máquina, Spark puede repartir el trabajo entre varias.

Recuerda los **tres lotes de mangos** de la presentación. Cada equipo calcula el peso y costo de su lote. Luego se reúnen los kilos y los costos para calcular el precio promedio por kilo de toda la compra.

Ahora repetiremos el promedio por mercado. En pandas usamos `groupby(...).mean()`. En Spark usamos `groupBy(...).avg(...)`.

In [0]:
# Convertimos las dos columnas que necesita este ejemplo.
papa_spark = spark.createDataFrame(
    papa_capira_antioquia[["mercado", "precio_promedio_cop"]]
)

# Calculamos el mismo promedio por mercado con Spark.
promedio_spark = papa_spark.groupBy("mercado").avg("precio_promedio_cop")
promedio_spark = promedio_spark.withColumnRenamed(
    "avg(precio_promedio_cop)", "precio_promedio_cop"
)

display(promedio_spark.orderBy("precio_promedio_cop"))

**¿Cuál es la deiferencia entre resolver el ejercicio con pandas y con Spark?** 

In [0]:
print("Tipo de datos en pandas: ", type(promedio_periodo))
print("Tipo de datos en spark: ", type(promedio_spark))

## 5 · Data Challenge

### ¿Dónde comprarías limón Tahití en el Valle del Cauca?

Un negocio de bebidas debe elegir **un solo mercado** para comprar **500 kg de limón Tahití al mes**. Analiza si el mercado más barato en enero habría sido también el de menor costo al comprarle de enero a octubre de 2026.

1. **Crea dos gráficas de barras.** Una con el precio de enero y otra con el promedio de enero a octubre. 
2. **Escribe dos hallazgos verificables.** Identifica el mercado más económico de enero y el más económico del periodo, con el precio que respalda cada elección.
3. **Calcula la diferencia de gasto.** Compara comprar siempre en el mercado elegido en enero con comprar siempre en el de menor costo del periodo. 
4. **Recomienda una acción.** Menciona el mercado que elegirías y la evidencia que respalda tu recomendación.

**Pistas.** Puedes copiar las celdas del caso de la papa capira y adaptar el filtro a `"Limón Tahití"` y `"VALLE DEL CAUCA"`. Usa una variable nueva, por ejemplo `limon_valle`, para conservar lo que hiciste en clase.

In [0]:
# Reto 1. Filtra el producto, el departamento y enero. Muestra la tabla para tu primera gráfica.
# Puedes copiar y adaptar las celdas de la Pregunta 1.

In [0]:
# Reto 2. Calcula el promedio por mercado de enero a octubre y muestra la tabla.
# Puedes copiar y adaptar la celda de la Pregunta 2.

### Mi respuesta

Haz doble clic en esta celda de texto y completa las cuatro líneas. Incluye las unidades y usa tus gráficas para justificar la decisión.

- **En enero habría elegido:** …
- **Para enero a octubre, el menor costo habría sido en:** …
- **Diferencia estimada en COP para 500 kg al mes durante 10 meses:** …
- **Mi recomendación:** …

### Exporta y envía tu notebook

1. Guarda las dos gráficas del reto y completa **Mi respuesta**.
2. En Databricks, abre **File → Export…**, elige **HTML**, marca **Include outputs** y pulsa **Export**.
3. Abre el archivo en tu navegador y verifica que se vean las gráficas, los resultados y tu respuesta escrita. Si falta algo, corrígelo en el notebook y exporta de nuevo.
4. Renombra el archivo como `DataChallenge_Nombre_Apellido.html` y adjúntalo en un correo a **ana.oviedo@upb.edu.co**. 

---

**Fuente de los datos:** DANE–SIPSA, componente de precios mayoristas. Hoja **2026** del archivo `anex-SIPSA-SerieHistoricaMayorista-2026.xlsx`. El CSV del taller conserva el corte de enero a octubre de 2026, aunque el DANE publique actualizaciones posteriores.

[Series históricas mensuales de SIPSA](https://www.dane.gov.co/index.php/estadisticas-por-tema/agropecuario/sistema-de-informacion-de-precios-sipsa/mayoristas-boletin-mensual-1) · [Importar y exportar en Databricks](https://docs.databricks.com/aws/en/notebooks/notebook-export-import) · [Crear visualizaciones](https://docs.databricks.com/aws/en/visualizations/)